In [ ]:
using Pkg
repository = @__DIR__
while !isfile(joinpath(repository, "src", "Surge.jl"))
    parent = dirname(repository)
    parent == repository && error("Open this notebook from within the repository.")
    global repository = parent
end
Pkg.activate(joinpath(repository, "Starting_networks"))
include(joinpath(repository, "src", "Surge.jl"))
import .Surge
using JLD2
include(joinpath(repository, "src", "experiment_diagnostics.jl"))
include(joinpath(repository, "src", "plotting_utils.jl"))

In [ ]:
run_directory = joinpath(repository, "results", "reported", "sparse_adaptation", "runs",
    "full__balanced__uniform__both__noise0cm", "fold_1", "repeat_1")
dataset_file = joinpath(repository, "Data", "thesis_v4", "uniform.jld2")
pretrained_file = joinpath(repository, "Trained_models", "flux_correction_pretrained.bson")
output_directory = joinpath(repository, "results", "notebooks", "diagnostics")
forecast_hours = 72.0
interior_indices = collect(11:90)
interval_boundaries = [0.0, 12.0, 24.0, 48.0, 72.0]

In [ ]:
dataset = Surge.load_burgers_ensemble(basename(dataset_file);
    data_path=dirname(dataset_file), relative_times=true)
pretrained_model = Surge.load_model(pretrained_file)
adapted_model = Surge.load_model(joinpath(run_directory, "best_model.bson"))
saved_run = JLD2.load(joinpath(run_directory, "result.jld2"), "result")

In [ ]:
diagnostics = ExperimentDiagnostics.evaluate_models(
    (:adapted => adapted_model, :pretrained => pretrained_model),
    dataset, saved_run.split, saved_run.configuration.stations;
    forecast_hours=forecast_hours, interior_indices=interior_indices,
    interval_boundaries=interval_boundaries, keep_series=true)
Surge.save_table(joinpath(output_directory, "diagnostics", "metrics.tsv"), diagnostics.summaries)
for record in diagnostics.series
    Surge.save_table(joinpath(output_directory, "diagnostics", "$(record.model)_event_$(record.event).tsv"),
        record.series)
end
display(ExperimentPlots.plot_report_series(output_directory, diagnostics))